# ⚠️ InFi-Check — Structured Error Dataset Generation Pipeline

Notebook này thực hiện **Bước 5** trong pipeline InFi-Check:
```
new_reference/*_ref.json  →  [structured_dataset_gen]  →  short_error_dataset/<doc>/<error_type>/<method>.txt
```

| Bước | File | Mô tả |
|------|------|-------|
| 1 | `dataset_*.py` | Làm sạch data |
| 2 | ✅ | Upload document lên Drive |
| 3 | ✅ `summary_gen_pipeline.ipynb` | Sinh tóm tắt |
| 4 | ✅ `eval_and_reference_gen_pipeline.ipynb` | Refine + tìm câu hỗ trợ |
| **5** | **`structured_dataset_gen_pipeline.ipynb`** | **← Notebook này** |
| 6 | `prepare_dataset_base.py` | Xuất JSONL |

**Notebook này làm gì?**

Với mỗi document, nó dùng `gpt-4o-mini` để **sinh ra các tóm tắt có lỗi nhân tạo** dựa trên các instruction template trong `summary_gen_prompt/structured_extrinsic/`. Mỗi instruction tương ứng với một loại lỗi (Entity Error, Predicate Error, Circumstance Error...) và một phương pháp tạo lỗi (swapping, compressing, introducing extrinsic info...).

**Output:** `short_error_dataset/<doc_name>/<error_type>/<method>/<method>.txt`

> ⚠️ Bước này tốn nhiều API call nhất vì mỗi document × mỗi instruction type đều gọi API một lần.

## 1. Mount Google Drive & kiểm tra cấu trúc thư mục

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️ CHỈNH đường dẫn nếu cần
PROJECT_ROOT     = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset'
PROMPT_ROOT      = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/summary_gen_prompt'
BASE_DOC_FOLDER  = '/content/drive/MyDrive/Dataset NLP'

REFERENCE_FOLDER = os.path.join(PROJECT_ROOT, 'new_reference')
ERROR_FOLDER     = os.path.join(PROJECT_ROOT, 'short_error_dataset')
SUMMARY_ROOT     = os.path.join(PROJECT_ROOT, 'new_summary')

os.makedirs(ERROR_FOLDER, exist_ok=True)

# Build doc_name -> category từ new_summary subfolder
DOC_CATEGORY_MAP = {}
for cat in os.listdir(SUMMARY_ROOT):
    cat_path = os.path.join(SUMMARY_ROOT, cat)
    if not os.path.isdir(cat_path):
        continue
    for f in os.listdir(cat_path):
        if f.endswith('_summary.txt'):
            DOC_CATEGORY_MAP[f.replace('_summary.txt', '')] = cat

ref_files = [f for f in os.listdir(REFERENCE_FOLDER) if f.endswith('.json')]
done_docs = [d for d in os.listdir(ERROR_FOLDER) if os.path.isdir(os.path.join(ERROR_FOLDER, d))]

print(f'📂 Base doc folder   : {BASE_DOC_FOLDER}')
print(f'📂 Reference folder  : {REFERENCE_FOLDER}')
print(f'📂 Error dataset     : {ERROR_FOLDER}')
print(f'📂 Prompt folder     : {PROMPT_ROOT}')
print(f'🗂️  DOC_CATEGORY_MAP  : {len(DOC_CATEGORY_MAP)} entries')
print()
print(f'📄 Reference JSON    : {len(ref_files)}')
print(f'✅ Doc đã có error   : {len(done_docs)}')
print(f'⏳ Doc chưa xử lý   : {len(ref_files) - len(done_docs)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Base doc folder   : /content/drive/MyDrive/Dataset NLP
📂 Reference folder  : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_reference
📂 Error dataset     : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/short_error_dataset
📂 Prompt folder     : /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/summary_gen_prompt
🗂️  DOC_CATEGORY_MAP  : 647 entries

📄 Reference JSON    : 647
✅ Doc đã có error   : 134
⏳ Doc chưa xử lý   : 513


## 2. Cài đặt thư viện

In [2]:
!pip install -q openai

## 3. Cấu hình API key

> 🔑 Bước này chỉ cần **1 API key** cho `gpt-4o-mini` (OpenAI).  
> Dùng Colab Secrets: `Tools → Secrets → Add new secret` tên `OPENAI_API_KEY`.

In [3]:
from google.colab import userdata

# --- Cách 1: Colab Secrets (khuyến nghị) ---
OPENROUTER_API_KEY5 = userdata.get('OPENROUTER_API_KEY5')

# --- Cách 2: Paste trực tiếp (chỉ dùng khi test) ---
# OPENAI_API_KEY = 'sk-proj-xxx'

def _mask(k):
    return f'{k[:8]}...{k[-4:]}' if k and len(k) > 12 else '❌ CHƯA CÓ'

print(f'OpenAI key: {_mask(OPENROUTER_API_KEY5)}')

OpenAI key: sk-or-v1...42c9


## 4. Đọc và parse các instruction template

Các file `.txt` trong `summary_gen_prompt/structured_extrinsic/` định nghĩa từng loại lỗi và cách tạo ra nó.  
Mỗi file có cấu trúc:
```
Error Type: ...
Method: ...
Few Shot: Yes/No
Instruction: ...
Format: ...
[Example Document / Summary / Output nếu Few Shot = Yes]
```

In [4]:
import re


def parse_txt_to_dict(txt_content: str) -> dict | None:
    """Parse một file instruction .txt thành dict."""
    result = {
        'Error Type': '', 'Method': '',
        'Instruction': '', 'Format': '', 'Few Shot': False
    }

    for field, pattern in [
        ('Error Type', r'Error Type: (.*)'),
        ('Method',     r'Method: (.*)'),
    ]:
        m = re.search(pattern, txt_content)
        if m:
            result[field] = m.group(1).strip()
        else:
            print(f'  [WARN] Không parse được {field}')
            return None

    m = re.search(r'Few Shot: (.*)', txt_content)
    if m:
        fs = m.group(1).strip()
        assert fs in ['Yes', 'No'], f'Few Shot phải là Yes/No, nhận được: {fs}'
        result['Few Shot'] = (fs == 'Yes')
    else:
        print('  [WARN] Không parse được Few Shot')
        return None

    m = re.search(r'Instruction:([\s\S]*?)Format:', txt_content)
    if m:
        result['Instruction'] = m.group(1).strip()
    else:
        print('  [WARN] Không parse được Instruction')
        return None

    if result['Few Shot']:
        for field, pat_start, pat_end in [
            ('Format',          r'Format:([\s\S]*?)Example Document:',  None),
            ('Example Document',r'Example Document:([\s\S]*?)Example Summary:', None),
            ('Example Summary', r'Example Summary:([\s\S]*?)Example Output:', None),
        ]:
            m = re.search(pat_start, txt_content)
            if m:
                result[field] = m.group(1).strip()
            else:
                print(f'  [WARN] Không parse được {field}')
                return None
        m = re.search(r'Example Output:([\s\S]*)', txt_content)
        if m:
            result['Example Output'] = str(eval(m.group(1).strip()))
        else:
            print('  [WARN] Không parse được Example Output')
            return None
    else:
        m = re.search(r'Format:([\s\S]*)', txt_content)
        if m:
            result['Format'] = m.group(1).strip()
        else:
            print('  [WARN] Không parse được Format')
            return None

    return result


def load_instructions(prompt_root: str) -> dict:
    """
    Đọc toàn bộ file .txt trong prompt_root và parse thành dict.
    Key: '<Error Type>|<Method>'
    """
    instruction_dict = {}
    for root, dirs, files in os.walk(prompt_root):
        for fname in files:
            if not fname.endswith('.txt'):
                continue
            full_path     = os.path.join(root, fname)
            relative_path = os.path.relpath(full_path, prompt_root)

            # ĐÃ XÓA ĐIỀU KIỆN CHẶN THƯ MỤC Ở ĐÂY ĐỂ NÓ ĐỌC TẤT CẢ

            with open(full_path, 'r', encoding='utf-8') as f:
                txt = f.read()
            parsed = parse_txt_to_dict(txt)
            if not parsed:
                print(f'  [SKIP] Lỗi parse: {relative_path}')
                continue
            parsed['Relative Path'] = relative_path
            key = f"{parsed['Error Type']}|{parsed['Method']}"
            instruction_dict[key] = parsed
    return instruction_dict


# Load
INSTRUCTION_DICT = load_instructions(PROMPT_ROOT)

print(f'✅ Đã load {len(INSTRUCTION_DICT)} instruction type:')
for k in sorted(INSTRUCTION_DICT.keys()):
    fs = '(few-shot)' if INSTRUCTION_DICT[k]['Few Shot'] else ''
    print(f'   • {k} {fs}')

✅ Đã load 9 instruction type:
   • Extrinsic/Extrinsic/Extrinsic Information|Introducing extrinsic information 
   • Intrinsic/Discourse/Co-reference|Merging two sentences with diffrent subject (few-shot)
   • Intrinsic/Discourse/Co-reference|Swapping Pronouns (few-shot)
   • Intrinsic/Discourse/Discourse Link|Reverse Logical Relationship 
   • Intrinsic/Semantic Frame/Circumstance|Swapping numbers (few-shot)
   • Intrinsic/Semantic Frame/Entity|Compressing words 
   • Intrinsic/Semantic Frame/Entity|Swapping entities 
   • Intrinsic/Semantic Frame/Predicate|Modifying Predictions 
   • Intrinsic/Semantic Frame/Predicate|Swapping relation maskings 


## 5. Khởi tạo OpenAI client & hàm gọi API

In [5]:
import time
import json
from openai import OpenAI, BadRequestError
from requests.exceptions import RequestException

client = OpenAI(
    api_key  = OPENROUTER_API_KEY5,
    base_url = 'https://openrouter.ai/api/v1'
)

# ✏️  Đổi model nếu cần (gpt-4o-mini là rẻ + nhanh nhất)
GEN_MODEL = 'qwen/qwen-2.5-72b-instruct'


def build_messages(document: str, summary: str, instruction: dict) -> list:
    """Xây dựng danh sách messages cho API call."""
    inst_text = instruction['Instruction']
    fmt_text  = instruction['Format']
    vi_note   = (
        '\nNote: The document and summary are in Vietnamese. '
        'All modifications must be applied to the Vietnamese text and the result must remain in Vietnamese.'
    )

    sys_msg = (
        'You are a helpful assistant. '
        'The document and summary may be written in Vietnamese. '
        'When modifying the summary, keep all Vietnamese text in Vietnamese — do not translate.'
    )

    if instruction['Few Shot']:
        user_intro = (
            f'Here is a document with a summary (the summary is given as a Python list, '
            f'each element is a sentence string). '
            f'Please create a fake summary based on the original summary by the following steps:\n'
            f'{inst_text}{vi_note}\n'
            f'Make sure the new summary is NOT fully supported by the document, '
            f'and do not change any other part of the summary besides those associated with the modification.\n\n'
            f'You should only respond in the format described below. '
            f'Do not return anything else. START YOUR RESPONSE WITH \'{{\'.\n\n'
            f'Return the result as a Python dictionary with the following keys:\n{fmt_text}\n'
            f'Replace any line breaks in the values with \'\\n\' so that the dictionary can be parsed using eval().'
        )
        return [
            {'role': 'system',    'content': sys_msg},
            {'role': 'user',      'content': user_intro},
            {'role': 'assistant', 'content': 'Sure! Please give me the document and the summary.'},
            {'role': 'user',      'content': f"Document:\n{instruction['Example Document']}\n\nSummary:\n{instruction['Example Summary']}"},
            {'role': 'assistant', 'content': instruction['Example Output']},
            {'role': 'user',      'content': f'Document:\n{document}\n\nSummary:\n{summary}'},
        ]
    else:
        user_prompt = (
            f'Here is a document with a summary. '
            f'Please create a fake summary based on the original summary by the following steps:\n'
            f'{inst_text}{vi_note}\n'
            f'Make sure the new summary is NOT fully supported by the document, '
            f'and do not change any other part of the summary besides those associated with the modification.\n\n'
            f'Document:\n{document}\n\nSummary:\n{summary}\n\n'
            f'You should only respond in the format described below. '
            f'Do not return anything else. START YOUR RESPONSE WITH \'{{\'.\n\n'
            f'Return the result as a Python dictionary with the following keys:\n{fmt_text}\n\n'
            f'Make sure the dictionary can be parsed using eval():\n'
            f'Replace any line breaks in the values with \'\\n\'.\n'
            f'Wrap each string with double quotes ("), replace any double quotes (") inside a string with single quotes (\').'
        )
        return [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': user_prompt},
        ]


def call_api(messages: list) -> dict | str:
    """
    Gọi API và trả về dict kết quả.
    Trả về string 'FAIL TO GENERATE DATA' nếu thất bại.
    """
    answer = ''
    for attempt in range(3):
        try:
            resp   = client.chat.completions.create(
                model       = GEN_MODEL,
                messages    = messages,
                temperature = 0.00001
            )
            answer = resp.choices[0].message.content
            # Loại markdown code fence
            answer = answer.replace('```python', '').replace('```json', '').replace('```', '')
            start  = answer.find('{')
            end    = answer.rfind('}') + 1
            assert start >= 0 and end > start
            return eval(answer[start:end])

        except RequestException as e:
            wait = 2 ** attempt
            print(f'  [attempt {attempt+1}] network error: {e} — retry in {wait}s')
            time.sleep(wait)
        except BadRequestError as e:
            print(f'  ❌ BadRequest: {e}')
            return 'FAIL TO GENERATE DATA'
        except Exception as e:
            if "I notice you've shared" in str(answer):
                return 'FAIL TO GENERATE DATA'
            print(f'  [attempt {attempt+1}] error: {e}')

    return 'FAIL TO GENERATE DATA'


print(f'✅ Client khởi tạo xong — model: {GEN_MODEL}')

✅ Client khởi tạo xong — model: qwen/qwen-2.5-72b-instruct


## 6. Kiểm tra nhanh với 1 document trước khi chạy toàn bộ

In [6]:
if not INSTRUCTION_DICT:
    print('❌ Không có instruction nào — kiểm tra PROMPT_ROOT')
else:
    test_ref_name = None
    for fname in sorted(os.listdir(REFERENCE_FOLDER)):
        if not fname.endswith('.json'):
            continue
        doc_name  = fname.replace('_ref.json', '')
        error_dir = os.path.join(ERROR_FOLDER, doc_name)
        if not os.path.exists(error_dir):
            test_ref_name = fname
            break

    if test_ref_name is None:
        print('ℹ️  Tất cả document đã có error data rồi!')
    else:
        doc_name = test_ref_name.replace('_ref.json', '')
        print(f'🧪 Test document: {doc_name}\n')

        category = DOC_CATEGORY_MAP.get(doc_name, '')
        doc_path = os.path.join(BASE_DOC_FOLDER, category, f'{doc_name}.txt') if category else None

        if not doc_path or not os.path.exists(doc_path):
            print(f'⚠️  Không tìm thấy document: {doc_path}')
        else:
            with open(doc_path, 'r', encoding='utf-8') as f:
                test_doc = f.read().replace('"', "'")

            with open(os.path.join(REFERENCE_FOLDER, test_ref_name), 'r', encoding='utf-8') as f:
                ref_data = json.load(f)

            if ref_data.get('errors'):
                print('⚠️  Reference có lỗi — chọn document khác để test')
            else:
                sents = ref_data.get('find_support_result', [])
                test_summary = '[' + ', '.join(
                    '"' + s['summary sentence'].replace('"', "'") + '"'
                    for s in sents
                ) + ']'

                first_key  = next(iter(INSTRUCTION_DICT))
                first_inst = INSTRUCTION_DICT[first_key]
                print(f'Instruction type: {first_key}')
                print(f'Summary preview : {test_summary[:150]}...\n')

                messages = build_messages(test_doc, test_summary, first_inst)
                result   = call_api(messages)

                if isinstance(result, dict):
                    print('✅ API trả về thành công. Các key:')
                    for k, v in result.items():
                        print(f'   {k}: {str(v)[:80]}')
                else:
                    print(f'❌ Thất bại: {result}')


🧪 Test document: 'Đường_một_chiều_là_lối_thoát_kẹt_xe_cho_ngã_tư_Hàng_Xanh'

Instruction type: Intrinsic/Semantic Frame/Entity|Compressing words
Summary preview : ["Nút giao Hàng Xanh thường xuyên ùn tắc.", "Một thạc sĩ công trình giao thông kiến nghị áp dụng mô hình tương tự tại ngã tư Bảy Hiền cho bốn tuyến: Đ...

✅ API trả về thành công. Các key:
   original text in summary: Một thạc sĩ công trình giao thông kiến nghị áp dụng mô hình tương tự tại ngã tư 
   chosen element: Điện Biên Phủ, Xô Viết Nghệ Tĩnh, Bạch Đằng, Đinh Bộ Lĩnh
   modification explanation: Uses the method of removing or altering an essential part inside an entity by si
   modified element: Điện Biên Phủ
   modified text: Một thạc sĩ công trình giao thông kiến nghị áp dụng mô hình tương tự tại ngã tư 
   explanation: The modification changes the specific proposal from involving four streets to on
   full text of modified summary: ['Nút giao Hàng Xanh thường xuyên ùn tắc.', 'Một thạc sĩ công trình gia

## 6.5 Domain distribution & cap

> ⚙️ Chạy trước Bước 7 để đảm bảo không domain nào chiếm > 30%.
> Kết quả được lưu vào `ALLOWED_FILES` — Bước 7 tự động dùng biến này.


In [7]:
import random as _r

# Domain mapping đầy đủ theo category folder
DOMAIN_MAP = {
    'Cong_nghe': 'tech',          'So_hoa': 'tech',
    'Kinh_doanh': 'business',     'Kinh_te': 'business',
    'Bat_dong_san': 'business',   'Thi_truong': 'business',
    'Phap_luat': 'legal',         'Chinh_tri': 'politics',
    'Cong_doan': 'politics',
    'Doi_song': 'lifestyle',      'Du_lich': 'lifestyle',
    'Tam_su': 'lifestyle',        'Y_kien': 'lifestyle',
    'Suc_khoe': 'health',         'Y_te': 'health',
    'Giai_tri': 'entertainment',  'Truyen_hinh': 'entertainment',
    'Van_hoa': 'culture',
    'The_thao': 'sports',         'Oto_xe_may': 'sports',
    'The_gioi': 'world',          'Xa_Hoi': 'society',
    'Thoi_su': 'society',         'Chong_Dien_Bien_Hoa_Binh': 'society',
    'Khoa_hoc': 'science',        'Moi_truong': 'science',
    'Giao_duc': 'education',
}

# Build doc_name -> category từ new_summary subfolder
SUMMARY_ROOT     = os.path.join(PROJECT_ROOT, 'new_summary')
DOC_CATEGORY_MAP = {}   # doc_name -> category
for cat in os.listdir(SUMMARY_ROOT):
    cat_path = os.path.join(SUMMARY_ROOT, cat)
    if not os.path.isdir(cat_path):
        continue
    for f in os.listdir(cat_path):
        if f.endswith('_summary.txt'):
            doc_name = f.replace('_summary.txt', '')
            DOC_CATEGORY_MAP[doc_name] = cat

def get_domain(doc_name: str) -> str:
    cat = DOC_CATEGORY_MAP.get(doc_name, '')
    return DOMAIN_MAP.get(cat, 'other')

print(f'DOC_CATEGORY_MAP: {len(DOC_CATEGORY_MAP)} entries')

MAX_DOMAIN_RATIO = 0.30  # tối đa 30% mỗi domain

all_refs    = sorted([f for f in os.listdir(REFERENCE_FOLDER) if f.endswith('.json')])
total       = len(all_refs)
max_per_dom = int(total * MAX_DOMAIN_RATIO)
domain_files = {}
for f in all_refs:
    d = get_domain(f.replace('_ref.json', ''))
    domain_files.setdefault(d, []).append(f)

print(f'Tong: {total} files | Cap: {max_per_dom}/domain ({MAX_DOMAIN_RATIO*100:.0f}%)')
ALLOWED_FILES = set()
for domain, files in sorted(domain_files.items(), key=lambda x: -len(x[1])):
    count = len(files)
    flag  = '  [cap]' if count > max_per_dom else ''
    print(f'  {domain:<15} {count:>4}  ({count/total*100:.1f}%){flag}')
    _r.seed(42); _r.shuffle(files)
    for f in files[:max_per_dom]:
        ALLOWED_FILES.add(f)

print(f'\nSe xu ly: {len(ALLOWED_FILES)}/{total} files')


DOC_CATEGORY_MAP: 647 entries
Tong: 647 files | Cap: 194/domain (30%)
  lifestyle        285  (44.0%)  [cap]
  science          210  (32.5%)  [cap]
  culture          116  (17.9%)
  business          18  (2.8%)
  tech              12  (1.9%)
  politics           3  (0.5%)
  society            3  (0.5%)

Se xu ly: 540/647 files


## 7. Chạy toàn bộ dataset

> ⚡ **Tip Colab timeout:** Nếu session hết giờ giữa chừng, chỉ cần **Run All** lại —  
> file đã sinh sẽ được bỏ qua tự động nhờ cơ chế resume.
>
> ⚙️ Đặt `LIMIT_DOCS = 20` để xử lý 20 document mỗi lần nếu lo hết quota.
>
> 📊 Ước tính API call: `số_doc × số_instruction_type` (hiện tại mỗi doc cần ~`len(INSTRUCTION_DICT)` call)

In [ ]:
import time as _time

LIMIT_DOCS  = None    # ✏️ None = xử lý tất cả; đặt số nguyên để test (vd: 20)
EXIST_ERROR = ['Paul Warnke']  # doc_name có lỗi parse đã biết — bỏ qua

ref_files_sorted = sorted([
    f for f in os.listdir(REFERENCE_FOLDER)
    if f.endswith('.json') and f in ALLOWED_FILES
])

processed = 0
skipped   = 0
failed    = 0

print(f"📋 Sẽ xử lý tối đa {LIMIT_DOCS or len(ref_files_sorted)} / {len(ref_files_sorted)} documents")
print(f"🔧 Số instruction type: {len(INSTRUCTION_DICT)}\n")

for ref_fname in ref_files_sorted:
    if LIMIT_DOCS and processed >= LIMIT_DOCS:
        break

    doc_name = ref_fname.replace('_ref.json', '')
    if doc_name in EXIST_ERROR:
        print(f"[SKIP] {doc_name} — trong danh sách lỗi parse")
        skipped += 1
        continue

    # --- Tìm file document ---
    category = DOC_CATEGORY_MAP.get(doc_name, '')
    doc_path = os.path.join(BASE_DOC_FOLDER, category, f'{doc_name}.txt') if category else ''
    if not doc_path or not os.path.exists(doc_path):
        print(f"[SKIP] {doc_name} — không tìm thấy document ({doc_path})")
        skipped += 1
        continue

    with open(doc_path, 'r', encoding='utf-8') as f:
        document = f.read().replace('"', "\'")

    # --- Đọc reference ---
    with open(os.path.join(REFERENCE_FOLDER, ref_fname), 'r', encoding='utf-8') as f:
        ref_data = json.load(f)

    if ref_data.get('errors'):
        print(f"[SKIP] {doc_name} — reference có lỗi")
        skipped += 1
        continue

    sents = ref_data.get('find_support_result', [])
    summary = '[' + ', '.join(
        '"' + s['summary sentence'].replace('"', "\'") + '"'
        for s in sents
    ) + ']'

    # --- Gọi API cho từng instruction type ---
    any_new = False
    for inst_key, instruction in INSTRUCTION_DICT.items():
        out_dir  = os.path.join(
            ERROR_FOLDER, doc_name,
            instruction['Relative Path'].replace('.txt', '')
        )
        out_file = os.path.join(out_dir, instruction['Method'] + '.txt')

        if os.path.exists(out_file):
            continue  # đã có → bỏ qua

        messages = build_messages(document, summary, instruction)
        result   = call_api(messages)

        if not isinstance(result, dict):
            print(f"  ❌ {doc_name} | {inst_key} — {result}")
            failed += 1
            continue

        os.makedirs(out_dir, exist_ok=True)
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(str(result))
        any_new = True

    if any_new:
        processed += 1
        print(f"[{processed:>4}] ✅ {doc_name}")
    else:
        skipped += 1

print(f"\n{'='*50}")
print(f"✅ Xử lý mới : {processed}")
print(f"⏭️  Bỏ qua   : {skipped}")
print(f"❌ Thất bại  : {failed}")


📋 Sẽ xử lý tối đa 540 / 540 documents
🔧 Số instruction type: 9

[SKIP] 300_nhà_khoa_học_6_châu_lục_đến_Trường_Tài_năng_UEH.ISB_mở_hội_thảo_quốc_tế — reference có lỗi
[SKIP] 7_thiên_tài_toán_học_không_học_toán_ở_đại_học — reference có lỗi
  [attempt 1] error: invalid syntax. Perhaps you forgot a comma? (<string>, line 3)
  [attempt 2] error: invalid syntax. Perhaps you forgot a comma? (<string>, line 3)
  [attempt 3] error: invalid syntax. Perhaps you forgot a comma? (<string>, line 3)
  ❌ Bạn_trai_bỏ_mặc_trong_lúc_tôi_nghi_ngờ_có_thai | Intrinsic/Discourse/Co-reference|Merging two sentences with diffrent subject — FAIL TO GENERATE DATA
[SKIP] Bị_du_khách_&amp;apos;tố&amp;apos;_cân_thiếu_hải_sản_ở_TPHCM_Tiểu_thương_nói_gì — reference có lỗi
[SKIP] Cặp_vợ_chồng_tiến_sĩ_tặng_trường_cũ_12_triệu_USD_để_&amp;apos;tri_ân_những_người — reference có lỗi
[SKIP] Du_khách_nước_ngoài_bay_flycam_trái_phép_ven_bi

## 8. Kiểm tra cấu trúc output

In [ ]:
import random

# Đếm tổng số file đã sinh
total_error_files = 0
doc_dirs = [d for d in os.listdir(ERROR_FOLDER) if os.path.isdir(os.path.join(ERROR_FOLDER, d))]

for doc_dir in doc_dirs:
    for root, dirs, files in os.walk(os.path.join(ERROR_FOLDER, doc_dir)):
        total_error_files += len([f for f in files if f.endswith('.txt')])

print(f'📁 Số document có error data : {len(doc_dirs)}')
print(f'📄 Tổng số error file sinh ra: {total_error_files}\n')

# In thử nội dung 1 file ngẫu nhiên
if doc_dirs:
    sample_doc = random.choice(doc_dirs)
    sample_files = []
    for root, dirs, files in os.walk(os.path.join(ERROR_FOLDER, sample_doc)):
        for f in files:
            if f.endswith('.txt'):
                sample_files.append(os.path.join(root, f))

    if sample_files:
        sample_file = random.choice(sample_files)
        with open(sample_file, 'r', encoding='utf-8') as f:
            content = eval(f.read())

        rel = os.path.relpath(sample_file, ERROR_FOLDER)
        print(f'📄 Mẫu ngẫu nhiên: {rel}')
        print(f'   Error Type     : {content.get("error type", content.get("Error Type", "N/A"))}')
        print(f'   Wrong info     : {str(content.get("wrong information", ""))[:120]}')
        print(f'   Modified text  : {str(content.get("modified text", ""))[:120]}')

## 9. Kiểm tra document nào còn thiếu error file

In [ ]:
n_inst = len(INSTRUCTION_DICT)
incomplete_docs = []

for ref_fname in sorted(os.listdir(REFERENCE_FOLDER)):
    if not ref_fname.endswith('.json'):
        continue
    doc_name = ref_fname.replace('_ref.json', '')

    # Đếm số instruction đã có file
    done_count = 0
    for inst_key, instruction in INSTRUCTION_DICT.items():
        out_dir  = os.path.join(ERROR_FOLDER, doc_name,
                                instruction['Relative Path'].replace('.txt', ''))
        out_file = os.path.join(out_dir, instruction['Method'] + '.txt')
        if os.path.exists(out_file):
            done_count += 1

    if done_count < n_inst:
        incomplete_docs.append((doc_name, done_count, n_inst))

if not incomplete_docs:
    print(f'✅ Tất cả {len(os.listdir(REFERENCE_FOLDER))} document đã đủ {n_inst} error file!')
else:
    print(f'⚠️  {len(incomplete_docs)} document chưa đủ error file (cần {n_inst} mỗi doc):')
    for doc_name, done, total in incomplete_docs[:20]:
        print(f'   • {doc_name}: {done}/{total}')
    if len(incomplete_docs) > 20:
        print(f'   ... và {len(incomplete_docs)-20} document khác')
    print('\n👉 Chạy lại cell 7 để tiếp tục sinh error data còn thiếu.')

## ✅ Hoàn thành — Bước tiếp theo

Sau khi chạy xong, thư mục output đã có đủ:

```
selected_dataset/
  └── short_error_dataset/
        ├── <doc_name_1>/
        │     └── structured_extrinsic/<error_type>/<method>/<method>.txt
        ├── <doc_name_2>/
        │     └── ...
        └── ...
```

Chạy tiếp **Bước 6** — xuất dataset JSONL cuối cùng:

```bash
python prepare_dataset_base.py
```

Output sẽ nằm trong `training_dataset_construct/sft_dataset/jsonl/`:
- `summary_sft_train_pos1neg5_with_ref.jsonl`
- `summary_sft_valid_with_ref.jsonl`
- `summary_sft_test_with_ref.jsonl`